# Foundations of Data Science
## Tutorial 6 — Linear Regression · **Solutions**

Worked solutions for all `# your code here` exercises in `tutorial06_linear_regression.ipynb`. The non-exercise cells are identical so you can run this end-to-end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (9, 4)

## 1. The setup

$n$ training examples $(x_i, y_i)$ stacked as $X \in \mathbb{R}^{n\times d}$ and $y \in \mathbb{R}^n$. We want $\hat y = Xw$ with bias absorbed as an extra column of ones in $X$.

## 2. Ordinary Least Squares via SVD

$w^\star = V \Sigma^+ U^\top y$ where $X = U\Sigma V^\top$ is the SVD.

In [ ]:
def fit_ols(X, y):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return Vt.T @ ((U.T @ y) / s)

In [ ]:
n = 50
x = np.linspace(0, 10, n)
y_true = 2 * x + 1
y      = y_true + rng.normal(0, 1.5, size=n)

X = np.column_stack([np.ones(n), x])
w = fit_ols(X, y)
print(f"learned: bias = {w[0]:.3f}, slope = {w[1]:.3f}")
print(f"true:    bias = 1.000, slope = 2.000")

In [ ]:
x_grid = np.linspace(0, 10, 100)
plt.figure()
plt.scatter(x, y, alpha=0.7, label="data")
plt.plot(x_grid, 2*x_grid + 1, "k--", label="truth")
plt.plot(x_grid, w[0] + w[1]*x_grid, "r-", label="OLS fit")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("Simple linear regression")
plt.show()

### Exercise 6.0 — residuals · **Solution**

Plot residuals $r_i = y_i - \hat y_i$ versus $x$. They should look like white noise around 0.

**Why does the mean of the residuals have to be ≈ 0?** Because our design matrix includes a column of ones (the bias), the gradient of the loss w.r.t. the bias is $-2 \sum_i r_i$. At the optimum this gradient is zero, forcing $\sum_i r_i = 0$ exactly.

In [ ]:
y_hat = X @ w
residuals = y - y_hat

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, residuals, alpha=0.7)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("x"); axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs x")

axes[1].hist(residuals, bins=15, color="steelblue", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("residual"); axes[1].set_ylabel("count")
axes[1].set_title("Histogram of residuals")
plt.tight_layout()
plt.show()

print(f"mean of residuals: {residuals.mean():.2e}   (should be ≈ 0)")
print(f"std of residuals : {residuals.std():.3f}    (≈ noise std 1.5)")

## 3. Polynomial regression

In [ ]:
def polynomial_features(x, degree):
    return np.column_stack([x ** k for k in range(degree + 1)])

In [ ]:
n_train = 15
x_train = np.linspace(0, 2*np.pi, n_train)
y_train = np.sin(x_train) + rng.normal(0, 0.2, n_train)
x_dense = np.linspace(0, 2*np.pi, 300)
y_true  = np.sin(x_dense)

In [ ]:
degrees = [1, 3, 9, 14]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, deg in zip(axes, degrees):
    X_train = polynomial_features(x_train, deg)
    w = fit_ols(X_train, y_train)
    y_pred = polynomial_features(x_dense, deg) @ w
    ax.scatter(x_train, y_train, color="black", zorder=5)
    ax.plot(x_dense, y_true, "k--", alpha=0.6)
    ax.plot(x_dense, y_pred, "r-", lw=2)
    ax.set_ylim(-2, 2)
    ax.set_title(f"degree = {deg}")
plt.suptitle("Polynomial regression on noisy sin(x)", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
n_test = 50
x_test = rng.uniform(0, 2*np.pi, n_test)
y_test = np.sin(x_test) + rng.normal(0, 0.2, n_test)

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

degrees = range(1, 15)
train_errors, test_errors = [], []
for deg in degrees:
    X_tr = polynomial_features(x_train, deg)
    X_te = polynomial_features(x_test,  deg)
    w = fit_ols(X_tr, y_train)
    train_errors.append(mse(y_train, X_tr @ w))
    test_errors.append( mse(y_test,  X_te @ w))

plt.figure()
plt.semilogy(degrees, train_errors, "o-", label="train MSE")
plt.semilogy(degrees, test_errors,  "s-", label="test MSE")
plt.xlabel("polynomial degree"); plt.ylabel("MSE (log scale)")
plt.title("Train error keeps falling — test error U-shapes"); plt.legend()
plt.show()

### Model selection with AIC and BIC

When we *don't* have a held-out test set, **information criteria** estimate generalization from training data alone by penalizing the in-sample residual with a complexity term.

For a Gaussian linear model with $k$ weight parameters fit on $n$ training points:

$$ \mathrm{AIC} = n \log\hat\sigma^2 + 2k, \qquad \mathrm{BIC} = n \log\hat\sigma^2 + k \log n $$

where $\hat\sigma^2 = \mathrm{SSR}/n$. Lower is better. BIC penalizes complexity more heavily ($k\log n > 2k$ once $n>7$), so BIC tends to favor simpler models.

In [ ]:
n = n_train
degs = list(range(1, 8))         # cap below n-1 = 14 — see caveat below
aics, bics = [], []
for deg in degs:
    X_tr = polynomial_features(x_train, deg)
    w    = fit_ols(X_tr, y_train)
    ssr  = np.sum((y_train - X_tr @ w) ** 2)
    k    = deg + 1
    sigma2 = ssr / n
    aics.append(n * np.log(sigma2) + 2 * k)
    bics.append(n * np.log(sigma2) + k * np.log(n))

best_aic = degs[int(np.argmin(aics))]
best_bic = degs[int(np.argmin(bics))]

plt.figure()
plt.plot(degs, aics, "o-", label="AIC")
plt.plot(degs, bics, "s-", label="BIC")
plt.axvline(best_aic, color="C0", linestyle="--", alpha=0.5, label=f"AIC min @ deg {best_aic}")
plt.axvline(best_bic, color="C1", linestyle="--", alpha=0.5, label=f"BIC min @ deg {best_bic}")
plt.xlabel("polynomial degree")
plt.ylabel("information criterion (lower = better)")
plt.title("AIC and BIC computed on the training set only")
plt.legend()
plt.show()

print(f"AIC picks degree {best_aic}")
print(f"BIC picks degree {best_bic}")

**Caveat:** for degree $\geq n - 1$ the polynomial interpolates the data, $\mathrm{SSR} \to 0$, and $\log\hat\sigma^2 \to -\infty$. The criteria then keep falling and falsely prefer the highest degree — their derivation assumes $k \ll n$ and irreducible error. With $n = 15$ we cap the sweep at degree 7 to stay in the regime where AIC/BIC are reliable.\n\nAIC/BIC are popular when refitting is expensive (classical statistics, ARIMA time series, GLMs). They rely on Gaussian + correctly-specified assumptions; cross-validation is the safer choice when those don't hold.

## 4. Ridge regression

In [ ]:
def fit_ridge(X, y, lam):
    d = X.shape[1]
    A = X.T @ X + lam * np.eye(d)
    return np.linalg.solve(A, X.T @ y)

In [ ]:
deg = 14
X_train = polynomial_features(x_train, deg)
X_dense = polynomial_features(x_dense, deg)

lambdas = [0, 1e-4, 1e-2, 1.0]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, lam in zip(axes, lambdas):
    w = fit_ridge(X_train, y_train, lam) if lam > 0 else fit_ols(X_train, y_train)
    ax.scatter(x_train, y_train, color="black", zorder=5)
    ax.plot(x_dense, y_true, "k--", alpha=0.6)
    ax.plot(x_dense, X_dense @ w, "r-", lw=2)
    ax.set_ylim(-2, 2)
    ax.set_title(f"λ = {lam}\n||w|| = {np.linalg.norm(w):.1f}")
plt.suptitle(f"Ridge regression, degree {deg} polynomial", y=1.04)
plt.tight_layout(); plt.show()

## 5. Bias-variance trade-off

In [ ]:
n_runs = 50
deg = 9
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lam in zip(axes, [0.0, 1e-3, 10.0]):
    all_preds = []
    for _ in range(n_runs):
        x_tr = rng.uniform(0, 2*np.pi, n_train)
        y_tr = np.sin(x_tr) + rng.normal(0, 0.2, n_train)
        X_tr = polynomial_features(x_tr, deg)
        X_de = polynomial_features(x_dense, deg)
        w = fit_ridge(X_tr, y_tr, lam) if lam > 0 else fit_ols(X_tr, y_tr)
        all_preds.append(X_de @ w)
        ax.plot(x_dense, X_de @ w, color="red", alpha=0.1)
    ax.plot(x_dense, np.mean(all_preds, axis=0), color="red", lw=2, label="mean fit")
    ax.plot(x_dense, y_true, "k--", lw=2, label="truth")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"λ = {lam}")
    ax.legend(fontsize=8)
plt.suptitle(f"50 fits on different training sets (degree {deg})", y=1.04)
plt.tight_layout(); plt.show()

## 6. Cross-validation

In [ ]:
def cross_val_score_ridge(X, y, lam, k=5, rng=None):
    rng = rng or np.random.default_rng(0)
    n = len(y)
    idx = rng.permutation(n)
    folds = np.array_split(idx, k)
    errors = []
    for i in range(k):
        val_idx   = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        w = fit_ridge(X[train_idx], y[train_idx], lam)
        errors.append(np.mean((y[val_idx] - X[val_idx] @ w) ** 2))
    return np.mean(errors)

In [ ]:
deg = 14
X_train = polynomial_features(x_train, deg)
X_test  = polynomial_features(x_test,  deg)

lambdas = np.logspace(-6, 2, 30)
cv_errors   = [cross_val_score_ridge(X_train, y_train, lam, k=5, rng=np.random.default_rng(1)) for lam in lambdas]
test_errors = []
for lam in lambdas:
    w = fit_ridge(X_train, y_train, lam)
    test_errors.append(mse(y_test, X_test @ w))

best_lam = lambdas[np.argmin(cv_errors)]

plt.figure()
plt.loglog(lambdas, cv_errors,   "o-", label="5-fold CV error")
plt.loglog(lambdas, test_errors, "s-", label="test error (held out)")
plt.axvline(best_lam, color="red", linestyle="--", label=f"best λ ≈ {best_lam:.1e}")
plt.xlabel("λ"); plt.ylabel("MSE")
plt.title("CV picks λ without seeing the test set"); plt.legend()
plt.show()
print(f"Best λ from CV: {best_lam:.3e}")

### Exercise 6.1 — fit with CV-selected λ · **Solution**

In [ ]:
# Fit ridge with the best λ from cross-validation
w_best = fit_ridge(X_train, y_train, best_lam)
y_pred_dense = polynomial_features(x_dense, deg) @ w_best
y_pred_test  = X_test @ w_best

plt.figure()
plt.scatter(x_train, y_train, color="black", zorder=5, label="train data")
plt.plot(x_dense, y_true,       "k--", label="truth (sin)")
plt.plot(x_dense, y_pred_dense, "r-",  lw=2, label=f"ridge fit, λ = {best_lam:.1e}")
plt.ylim(-2, 2)
plt.xlabel("x"); plt.ylabel("y")
plt.title(f"Degree-{deg} polynomial fit, ridge with CV-selected λ")
plt.legend(); plt.show()

print(f"test MSE with best λ : {mse(y_test, y_pred_test):.4f}")
w_ols14 = fit_ols(X_train, y_train)
print(f"test MSE with λ = 0   : {mse(y_test, X_test @ w_ols14):.4f}  (plain OLS — explodes!)")

## 7. California housing

In [ ]:
data = fetch_california_housing()
X_raw = data.data
y_raw = data.target
feature_names = data.feature_names
print(f"n = {X_raw.shape[0]}, d = {X_raw.shape[1]}")
print("features:", feature_names)

In [ ]:
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
X_full = np.column_stack([np.ones(len(X_std)), X_std])

n = len(X_full)
perm = rng.permutation(n)
n_tr = int(0.8 * n)
tr, te = perm[:n_tr], perm[n_tr:]
X_tr, X_te = X_full[tr], X_full[te]
y_tr, y_te = y_raw[tr], y_raw[te]
print(f"train: {X_tr.shape}, test: {X_te.shape}")

In [ ]:
w_ols = fit_ols(X_tr, y_tr)
print(f"train MSE: {mse(y_tr, X_tr @ w_ols):.4f}")
print(f"test  MSE: {mse(y_te, X_te @ w_ols):.4f}\n")
print("bias:", f"{w_ols[0]:+.3f}")
for name, wi in zip(feature_names, w_ols[1:]):
    print(f"  {name:12s}: {wi:+.3f}")

### Exercise 6.2 — ridge on California housing · **Solution**

In [ ]:
# Sweep λ and compute 5-fold CV error on the *training* set
lambdas = np.logspace(-3, 4, 20)
cv_errs_housing = [
    cross_val_score_ridge(X_tr, y_tr, lam, k=5, rng=np.random.default_rng(7))
    for lam in lambdas
]
best_lam_h = lambdas[np.argmin(cv_errs_housing)]

# Fit ridge with best λ on full training set, evaluate on test
w_ridge = fit_ridge(X_tr, y_tr, best_lam_h)
test_mse_ridge = mse(y_te, X_te @ w_ridge)
test_mse_ols   = mse(y_te, X_te @ w_ols)

plt.figure()
plt.semilogx(lambdas, cv_errs_housing, "o-")
plt.axvline(best_lam_h, color="red", linestyle="--", label=f"best λ = {best_lam_h:.2f}")
plt.xlabel("λ"); plt.ylabel("5-fold CV MSE")
plt.title("Ridge tuning on California housing")
plt.legend(); plt.show()

print(f"OLS   test MSE: {test_mse_ols:.5f}")
print(f"Ridge test MSE: {test_mse_ridge:.5f}   (best λ = {best_lam_h:.3f})")
print(f"improvement   : {(test_mse_ols - test_mse_ridge)/test_mse_ols*100:+.3f}%")

**Discussion.** The improvement from ridge here is **tiny** (often a fraction of a percent) — the test MSE is essentially the same as plain OLS.

Why? With $n \approx 16{,}500$ training samples and only $d = 9$ parameters, we are firmly in the regime where OLS is already well-conditioned and not overfitting. The variance of the OLS estimate is roughly $\sigma^2 (X^\top X)^{-1}$, which shrinks like $1/n$ — already negligible here.

**Where regularization actually shines:**
- $d$ comparable to or larger than $n$ (high-dimensional regression, genomics, text features)
- highly collinear features (where $X^\top X$ is near-singular)
- polynomial / kernel expansions where the effective dimension explodes

**Moral of the story:** don't reach for ridge as a reflex. Diagnose the problem first — if you have lots of data per parameter and well-conditioned features, plain OLS is fine.

## Takeaways

- OLS = $\|y - Xw\|^2$ minimization, solved cleanly via SVD.
- Polynomial features turn linear regression into curve fitting.
- Overfitting: train ↓, test ⌣.
- Ridge ($\|w\|^2$ penalty) tames overfitting; closed form.
- Bias-variance: capacity is a knob, not a virtue.
- CV picks λ honestly.
- Regularization helps most when $d$ is comparable to $n$.